In [4]:
class NoMaskGCNConv(GCNConv):
    def compute_mask(self, inputs, mask=None):
        return None

    def call(self, inputs, training=None, mask=None):
        # Explicitly discard mask
        return super().call(inputs, mask=None)
       
class GCN(tf.keras.Model):
    def __init__(self, n_labels, seed=42):
        super().__init__()
        initializer = tf.keras.initializers.GlorotUniform(seed=seed)
        self.conv1 = NoMaskGCNConv(16, activation='relu', kernel_initializer=initializer)
        self.conv2 = NoMaskGCNConv(n_labels, activation='softmax', kernel_initializer=initializer)

    def call(self, inputs, training=False):
        x, a = inputs
        intermediate_embeddings = self.conv1([x, a])
        x = self.conv2([intermediate_embeddings, a])
        return x, intermediate_embeddings

from spektral.layers import GATConv
import tensorflow as tf

# Define a custom wrapper for GATConv that avoids mask issues
class NoMaskGATConv(GATConv):
    def compute_mask(self, inputs, mask=None):
        return None

    def call(self, inputs, training=None, mask=None):
        # Explicitly discard the mask argument
        return super().call(inputs, mask=None)


# Define the GAT model using the NoMaskGATConv
class GAT(tf.keras.Model):
    def __init__(self, n_labels, num_heads=8, seed=42):
        super().__init__()
        initializer = tf.keras.initializers.GlorotUniform(seed=seed)

        # Use the custom NoMaskGATConv instead of the original GATConv
        self.conv1 = NoMaskGATConv(16, attn_heads=num_heads, concat_heads=True, activation='elu', kernel_initializer=initializer)
        self.conv2 = NoMaskGATConv(n_labels, attn_heads=1, concat_heads=False, activation='softmax', kernel_initializer=initializer)

    def call(self, inputs):
        x, a = inputs
        intermediate_embeddings = self.conv1([x, a])  # Store intermediate embeddings
        x = self.conv2([intermediate_embeddings, a])
        return x, intermediate_embeddings  # Return both final output and intermediate embeddings

# Define the GraphSAGE model
class GraphSAGE(tf.keras.Model):
    def __init__(self, n_labels, hidden_dim=16, aggregator='mean', seed=42):
        super().__init__()
        initializer = tf.keras.initializers.GlorotUniform(seed=seed)

        self.conv1 = GraphSageConv(hidden_dim, activation='relu', aggregator=aggregator, kernel_initializer=initializer)
        self.conv2 = GraphSageConv(n_labels, activation='softmax', aggregator=aggregator, kernel_initializer=initializer)

    def call(self, inputs):
        x, a = inputs
        intermediate_embeddings = self.conv1([x, a])  # Store intermediate embeddings
        x = self.conv2([intermediate_embeddings, a])
        return x, intermediate_embeddings  # Return both final output and intermediate embeddings

classifiers=['gcn','gat','graphsage']

In [3]:
import tensorflow as tf

In [29]:
import os
import random
import numpy as np
import pandas as pd
import torch
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torch_geometric.datasets import Planetoid, WikiCS, Amazon
import networkx as nx
import time

# Configuration
BASE_DIR = "."
#DATASETS = ["Cora", "CiteSeer", "PubMed", "WikiCS", "AmazonPhotos"]
DATASETS = ["AmazonPhotos"]
SEEDS = [42, 46, 123, 2025, 999]
CLASSIFIERS = ['gcn', 'gat', 'graphsage']
SPLITS = ['30_70', '70_30']
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
MASKS_DIR = os.path.join(BASE_DIR, "masks")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def convert_to_networkx(A):
    return nx.from_scipy_sparse_array(A)

def evaluate_model(true_labels, predicted_labels):
    accuracy = accuracy_score(true_labels, predicted_labels)
    f1 = f1_score(true_labels, predicted_labels, average='macro')
    cm = confusion_matrix(true_labels, predicted_labels)
    print(cm)
    return {'accuracy': accuracy, 'f1_score': f1}

for dataset_name in DATASETS:
    print(f"\n=== Processing dataset: {dataset_name} ===")

    dataset_results_dir = os.path.join(RESULTS_DIR, dataset_name)
    os.makedirs(dataset_results_dir, exist_ok=True)

    for split in SPLITS:
        print(f"\nProcessing split: {split}")

        split_results = []
        data_dir = os.path.join(BASE_DIR, "data", dataset_name)

        # Load dataset
        if dataset_name in ["Cora", "CiteSeer", "PubMed"]:
            dataset = Planetoid(root=data_dir, name=dataset_name)[0]
        elif dataset_name == "WikiCS":
            dataset = WikiCS(root=data_dir)[0]
        elif dataset_name == "AmazonPhotos":
            dataset = Amazon(root=data_dir, name='Photo')[0]  # fix: use 'Photo' with capital P
        else:
            raise NotImplementedError(f"Dataset {dataset_name} loader not implemented")

        ground_truth_labels = dataset.y.numpy()
        labels = ground_truth_labels  # Already integer class labels

        # Build adjacency matrix
        edge_index = dataset.edge_index.numpy()
        num_nodes = dataset.num_nodes
        A = sp.coo_matrix(
            (np.ones(edge_index.shape[1]), (edge_index[0], edge_index[1])),
            shape=(num_nodes, num_nodes)
        )
        X_full = dataset.x.numpy()

        for classifier in CLASSIFIERS:
            print(f"\nRunning classifier: {classifier.upper()}")

            all_accuracies = []
            all_f1_scores = []
            total_execution_time = 0.0

            for seed in SEEDS:
                print(f"\nSeed: {seed}")
                set_global_seed(seed)

                mask_file = os.path.join(MASKS_DIR, dataset_name, split,
                                         f"{dataset_name}_{split}_masked_indices_seed{seed}.npy")
                labels_to_be_masked = np.load(mask_file)

                masked_labels = np.array([
                    -1 if i in labels_to_be_masked else labels[i]
                    for i in range(len(labels))
                ])
                label_mask = masked_labels != -1

                emb_file = os.path.join(OUTPUT_DIR, "embeddings", dataset_name, split,
                                        f"{dataset_name}_{split}_seed{seed}_mnmf.pkl")
                embedding_matrix = pd.read_pickle(emb_file)

                start_time = time.time()
                result, _ = train_and_evaluate(
                    embedding_dict={classifier: embedding_matrix},
                    embedding=classifier,
                    classifier=classifier,
                    ground_truth_labels=dataset.y.numpy(),
                    masked_labels=masked_labels
                )
                end_time = time.time()
                duration = end_time - start_time
                total_execution_time += duration
                print(f"Seed {seed} execution time: {duration:.2f} seconds")

                all_accuracies.append(result['accuracy'])
                all_f1_scores.append(result['f1_score'])

            avg_accuracy = np.mean(all_accuracies)
            std_accuracy = np.std(all_accuracies)
            avg_f1 = np.mean(all_f1_scores)
            std_f1 = np.std(all_f1_scores)
            avg_time = total_execution_time / len(SEEDS)

            summary = {
                'classifier': classifier,
                'average_accuracy': f"{avg_accuracy:.4f} ± {std_accuracy:.4f}",
                'average_f1_score': f"{avg_f1:.4f} ± {std_f1:.4f}",
                'average_execution_time_sec': round(avg_time, 2)
            }
            split_results.append(summary)

        # Save results per split
        split_dir = os.path.join(dataset_results_dir, split)
        os.makedirs(split_dir, exist_ok=True)

        df_split = pd.DataFrame(split_results)
        filename = os.path.join(split_dir, f"{dataset_name}_analysis_results_{split}.csv")
        df_split.to_csv(filename, index=False)
        print(f"\nSaved results for split {split} at: {filename}")

    print(f"\n=== Dataset {dataset_name} completed ===\n")



=== Processing dataset: AmazonPhotos ===

Processing split: 30_70

Running classifier: GCN

Seed: 42
Epoch 20, Loss: 1.9806, Accuracy: 0.2537
Epoch 40, Loss: 1.6833, Accuracy: 0.3896
Epoch 60, Loss: 1.5538, Accuracy: 0.4112
Epoch 80, Loss: 1.4160, Accuracy: 0.4591
Epoch 100, Loss: 1.2314, Accuracy: 0.5706
Epoch 120, Loss: 0.9836, Accuracy: 0.6676
Epoch 140, Loss: 0.8322, Accuracy: 0.7047
Epoch 160, Loss: 0.7299, Accuracy: 0.7499
Epoch 180, Loss: 0.6656, Accuracy: 0.7897
Epoch 200, Loss: 0.6232, Accuracy: 0.7986
[[  5  19   1  71  11  25   2 107]
 [  6 639  11  75 109  27  68 244]
 [  3  65 130  27  57  14  94  93]
 [  6  42   5  86 286   6  34 189]
 [  1   6   1   2 373   8 176  34]
 [  4  31   4   0   0 265 116 135]
 [  5  60  37  23 698  17 432  69]
 [  2   2   0   1  90   4  19 104]]
Seed 42 execution time: 4.41 seconds

Seed: 46
Epoch 20, Loss: 1.9316, Accuracy: 0.2838
Epoch 40, Loss: 1.7720, Accuracy: 0.2905
Epoch 60, Loss: 1.4769, Accuracy: 0.5371
Epoch 80, Loss: 1.0289, Accurac

In [7]:
import scipy.sparse as sp

In [24]:
# ---- Define training and evaluation function ----
def train_and_evaluate(embedding_dict, embedding, classifier, ground_truth_labels, masked_labels):
    X = embedding_dict[embedding]
    num_emb_nodes = X.shape[0]  # number of nodes in embeddings

    # Subset labels to match embedding size
    masked_labels_sub = masked_labels[:num_emb_nodes]
    ground_truth_labels_sub = ground_truth_labels[:num_emb_nodes]
    train_mask = masked_labels_sub != -1

    # Training data
    X_train = X[train_mask]
    Y_train = ground_truth_labels_sub[train_mask]
    Y_train = tf.one_hot(Y_train, depth=len(np.unique(ground_truth_labels)))
    Y_train = tf.cast(Y_train, dtype='float32')

    # Build adjacency matrix for embedding nodes
    # Convert global adjacency A to CSR for safe slicing
    A_csr = A.tocsr()  # Ensure A is in CSR format
    A_sub = A_csr[:num_emb_nodes, :num_emb_nodes]  # slice embedding nodes
    A_train = A_sub[train_mask, :][:, train_mask]   # slice training nodes
    A_train_coo = A_train.tocoo()
    indices_train = np.column_stack((A_train_coo.row, A_train_coo.col))
    A_train_tensor = tf.sparse.SparseTensor(
        indices=indices_train,
        values=A_train_coo.data,
        dense_shape=A_train_coo.shape
    )
    A_train_tensor = tf.sparse.reorder(A_train_tensor)

    # Initialize model
    n_labels = Y_train.shape[1]
    if classifier == 'gcn':
        model = GCN(n_labels)
    elif classifier == 'gat':
        model = GAT(n_labels)
    elif classifier == 'graphsage':
        model = GraphSAGE(n_labels)
    else:
        raise ValueError(f"Unknown classifier {classifier}")

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
    loss_fn = tf.keras.losses.CategoricalCrossentropy()
    acc_metric = tf.keras.metrics.CategoricalAccuracy()

    # Training loop with prints every 20 epochs
    epochs = 200
    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            predictions, _ = model([X_train, A_train_tensor])
            loss = loss_fn(Y_train, predictions)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        if (epoch + 1) % 20 == 0:
            acc_metric.update_state(Y_train, predictions)
            print(f"Epoch {epoch+1}, Loss: {loss.numpy():.4f}, Accuracy: {acc_metric.result().numpy():.4f}")
            acc_metric.reset_state()

    # Full graph prediction (restricted to embedding nodes)
    A_full_coo = A_sub.tocoo()
    indices_full = np.column_stack((A_full_coo.row, A_full_coo.col))
    A_full_tensor = tf.sparse.SparseTensor(
        indices=indices_full,
        values=A_full_coo.data,
        dense_shape=A_full_coo.shape
    )
    A_full_tensor = tf.sparse.reorder(A_full_tensor)

    predictions, emb = model([X, A_full_tensor])
    predicted_labels = tf.argmax(predictions, axis=1).numpy()

    # Evaluate only masked nodes
    masked_indices = np.where(masked_labels_sub == -1)[0]
    predicted_masked = predicted_labels[masked_indices]
    true_masked = ground_truth_labels_sub[masked_indices]

    results = evaluate_model(true_masked, predicted_masked)
    results['model'] = classifier
    results['embedding'] = embedding

    return results, emb


In [13]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
